# Vaardigheden week 7: twee tabellen koppelen

Dit notebook hoort bij het vaardighedenspoor. Het gebruikt een verzonnen bakkerij met vier winkels, zodat de vaardigheden los blijven van het project van deze week.

De bakkerij houdt haar gegevens bij in aparte tabellen: één voor de winkels, één voor de verkoop per maand en één voor de broodprijs. De meeste vragen over de bakkerij hebben er twee tegelijk nodig. Je koppelt tabellen, voorspelt en telt de rijen van elke koppeling, betrapt een koppeling die zonder waarschuwing rijen kwijtraakt, en zet een tabel om die in de verkeerde vorm binnenkomt.

**Zo werk je ermee:** voer elke cel één keer uit, op volgorde. Schrijf vóór elke koppeling op hoeveel rijen je verwacht, en vergelijk je schatting daarna met de telling.

## 1. Drie tabellen

De cel hieronder maakt de drie tabellen van de bakkerij. De seed in `default_rng(77)` zorgt ervoor dat de willekeurige verkoopcijfers voor iedereen gelijk zijn.

- `shops` heeft één rij per winkel, met de wijk waarin de winkel ligt.
- `sales` heeft één rij per winkel per maand, met het aantal broden dat de winkel verkocht.
- `prices` heeft één rij per maand, met de prijs van een brood. Niemand heeft de prijs van december al ingevoerd.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(77)

shops = pd.DataFrame({"shop": ["S01", "S02", "S03", "S04"], "district": [1, 1, 2, 3]})

months = list(range(1, 13))
sales = pd.DataFrame([(s, m) for s in shops["shop"] for m in months], columns=["shop", "month"])
sales["loaves"] = rng.poisson(900, size=len(sales))

# één prijs per maand; december is nog niet ingevoerd
prices = pd.DataFrame({"month": months[:11], "price": np.round(np.linspace(2.00, 2.50, 11), 2)})

print("shops:", shops.shape, " sales:", sales.shape, " prices:", prices.shape)
sales.head()

## 2. Wat één rij betekent

Elke tabel heeft een granulariteit: het ding dat één rij beschrijft. In `shops` is één rij een winkel. In `sales` is één rij een winkel in een maand. In `prices` is één rij een maand.

De sleutel is de kolom, of de groep kolommen, die dat ding benoemt. Een koppeling zoekt rijen bij elkaar via hun sleutel, dus test de sleutel voordat je koppelt. Een sleutel die één keer hoort voor te komen, moet ook één keer voorkomen.

In [ ]:
print("Elke winkel komt één keer voor in shops:", shops["shop"].is_unique)
print("Elke maand komt één keer voor in prices:", prices["month"].is_unique)
print("Herhaalde paren van winkel en maand in sales:", sales.duplicated(["shop", "month"]).sum())

## 3. Een eerste koppeling

Om de verkochte broden per wijk op te tellen, heb je de wijk uit `shops` nodig naast elke rij van `sales`. Een left join houdt elke rij van de linkertabel (`sales`) en voegt de passende kolommen uit de rechtertabel (`shops`) toe.

Elke winkel komt één keer voor in `shops`, dus elke verkooprij vindt precies één partner. De gekoppelde tabel hoort daarom evenveel rijen te hebben als `sales`. Voorspel het aantal en voer dan de cel uit.

In [ ]:
with_district = sales.merge(shops, on="shop", how="left")
print("Rijen vóór de koppeling:", len(sales))
print("Rijen na de koppeling:", len(with_district))
assert len(with_district) == len(sales)
print()
print(with_district.groupby("district")["loaves"].sum())

## 4. Vier soorten koppelingen

Twee kleine tabellen maken de vier soorten goed zichtbaar. De eerste heeft verkochte broden voor maand 1 tot en met 5. De tweede heeft prijzen voor maand 3 tot en met 7. Alleen maand 3, 4 en 5 staan in allebei.

- Een **inner** join houdt alleen de rijen waarvan de sleutel in beide tabellen staat.
- Een **left** join houdt elke rij van de linkertabel, met lege plekken waar de rechtertabel geen match heeft.
- Een **right** join houdt elke rij van de rechtertabel.
- Een **outer** join houdt elke rij van beide tabellen.

Voorspel het aantal rijen voor elke soort voordat je de cel uitvoert.

In [ ]:
small_sales = pd.DataFrame({"month": [1, 2, 3, 4, 5], "loaves": [880, 905, 870, 930, 910]})
small_prices = pd.DataFrame({"month": [3, 4, 5, 6, 7], "price": [2.10, 2.15, 2.20, 2.25, 2.30]})

for how in ["inner", "left", "right", "outer"]:
    joined = small_sales.merge(small_prices, on="month", how=how)
    print(f"{how} join: {len(joined)} rijen, maanden {joined['month'].tolist()}")

Een outer join met `indicator=True` voegt een kolom `_merge` toe. Die zegt waar elke rij vandaan komt: `both`, `left_only` of `right_only`. Zo zie je het snelst wat een koppeling wel en niet kon koppelen.

In [ ]:
check = small_sales.merge(small_prices, on="month", how="outer", indicator=True)
print(check)
print()
print(check["_merge"].value_counts())

## 5. Lees code die een AI-assistent schreef

We vroegen een AI-assistent: *"Koppel de verkoop aan de prijzen en geef me de totale omzet van de bakkerij over het jaar."* De assistent schreef de cel hieronder. Voer hem uit en beslis dan of het antwoord klopt.

In [ ]:
# Geschreven door een AI-assistent
priced = sales.merge(prices, on="month")
priced["revenue"] = priced["loaves"] * priced["price"]
print("Totale omzet over het jaar:", f"{priced['revenue'].sum():,.2f}")

De code draait zonder foutmelding, en het totaal ziet er redelijk uit. Tel nu de rijen. `merge` doet een inner join als je niet om een andere soort vraagt, en een inner join houdt alleen de rijen die in beide tabellen een match hebben.

In [ ]:
print("Rijen vóór de koppeling:", len(sales))
print("Rijen na de koppeling:", len(priced))
print("Maanden in het resultaat:", priced["month"].nunique())

Vier rijen verdwenen zonder waarschuwing. Het zijn de decemberverkopen van alle vier de winkels, en ze verdwenen omdat `prices` geen rij voor december heeft. Het totaal van de AI beslaat elf maanden, terwijl het label een jaar noemt.

De volgende cel doet de koppeling zorgvuldig. Een left join houdt elke verkooprij. De indicator laat zien welke rijen geen prijs vonden. Het argument `validate` laat pandas stoppen als een maand twee keer in `prices` staat.

In [ ]:
careful = sales.merge(prices, on="month", how="left", indicator=True, validate="many_to_one")
no_price = careful[careful["_merge"] == "left_only"]
print("Rijen zonder prijs:", len(no_price))
print("Broden verkocht in december zonder prijs:", f"{no_price['loaves'].sum():,}")
careful["revenue"] = careful["loaves"] * careful["price"]
print("Omzet van januari tot en met november:", f"{careful['revenue'].sum():,.2f}")

# CHECK: elke verkooprij overleeft de zorgvuldige koppeling, en alleen december mist een prijs
assert len(careful) == len(sales)
assert set(no_price["month"]) == {12}
print("CHECK geslaagd")

De omzet is hetzelfde getal dat de AI afdrukte. Het verschil is dat je nu weet wat het getal omvat, en dat je dat kunt zeggen: de omzet van januari tot en met november, terwijl de verkoop van december nog op een prijs wacht. Een rapport dat dit zegt, is eerlijk. Een rapport dat hetzelfde getal de omzet van een jaar noemt, misleidt de lezer.

## 6. Als een sleutel herhaald wordt

Stel dat iemand de prijs van juni verbetert door een nieuwe rij toe te voegen in plaats van de oude te wijzigen. Juni staat nu twee keer in de prijstabel, en elke verkooprij van juni vindt twee partners. De koppeling houdt ze allebei, dus juni telt twee keer mee.

In [ ]:
prices_twice = pd.concat([prices, pd.DataFrame({"month": [6], "price": [2.30]})], ignore_index=True)
doubled = sales.merge(prices_twice, on="month", how="left")
doubled["revenue"] = doubled["loaves"] * doubled["price"]
print("Rijen vóór de koppeling:", len(sales))
print("Rijen na de koppeling met juni twee keer:", len(doubled))
print("Omzet met juni twee keer geteld:", f"{doubled['revenue'].sum():,.2f}")

Het aantal rijen verraadt de fout opnieuw: er kwamen meer rijen uit dan erin gingen. Een left join op een sleutel die rechts één keer voorkomt, kan nooit rijen toevoegen, dus extra rijen betekenen altijd een herhaalde sleutel.

`validate="many_to_one"` legt die belofte vast in code: veel verkooprijen per maand, één prijs per maand. Als de prijstabel de belofte breekt, stopt pandas in plaats van te gokken.

In [ ]:
try:
    sales.merge(prices_twice, on="month", how="left", validate="many_to_one")
except pd.errors.MergeError as err:
    print("pandas stopte de koppeling:", err)

Het herstel hoort in de prijstabel thuis. Iemand moet beslissen welke prijs voor juni klopt. Hier houden we de latere rij en koppelen we opnieuw.

In [ ]:
prices_fixed = prices_twice.drop_duplicates("month", keep="last")
fixed = sales.merge(prices_fixed, on="month", how="left", validate="many_to_one")
print("Rijen na de koppeling met de herstelde prijzen:", len(fixed))

## 7. Sleutels die er hetzelfde uitzien maar niet passen

Een koppeling vergelijkt sleutels exact. Twee sleutels die voor een mens gelijk lijken, kunnen voor de computer op drie gewone manieren verschillen: een spatie aan het eind, een ander gebruik van hoofdletters, of een getal dat als tekst is opgeslagen.

Een manager typte deze tabel met bakkers per winkel met de hand. Tel wat er past.

In [ ]:
bakers = pd.DataFrame({"shop": ["S01", "s02", "S03 ", "S04"], "bakers": [5, 3, 4, 2]})
matched = shops.merge(bakers, on="shop", how="inner")
print("Winkels:", len(shops), " rijen in bakers:", len(bakers), " gekoppeld:", len(matched))
print("codes in shops: ", shops["shop"].tolist())
print("codes in bakers:", bakers["shop"].tolist())

Twee winkels verdwenen uit de inner join. De code `s02` is in kleine letters geschreven, en `S03 ` eindigt op een spatie. Maak de codes in beide tabellen op dezelfde manier schoon voordat je koppelt.

In [ ]:
bakers["shop"] = bakers["shop"].str.strip().str.upper()
matched = shops.merge(bakers, on="shop", how="inner")
print("Gekoppeld na het schoonmaken van de codes:", len(matched))

Een getal dat als tekst is opgeslagen, faalt luidruchtiger. Veel webbronnen sturen elke waarde als tekst, dus een maand kan binnenkomen als `"6"` in plaats van `6`. pandas weigert een tekstsleutel aan een getalsleutel te koppelen. Lees de melding tot het eind.

In [ ]:
prices_text = prices.astype({"month": str})      # de maand als tekst, zoals veel webbronnen hem sturen
try:
    sales.merge(prices_text, on="month", how="left")
except ValueError as err:
    print("pandas weigerde:", err)

De melding stelt `pd.concat` voor, en dat is hier het verkeerde gereedschap. `concat` stapelt tabellen, of zet ze naast elkaar op rijpositie, en kijkt nooit naar de sleutel. De cel hieronder laat de schade zien in een klein geval: vier maanden verkoop vanaf de eerste maand, naast vier prijzen die van de laatste maand terug zijn gesorteerd.

In [ ]:
first_rows = sales[sales["shop"] == "S01"].head(4).reset_index(drop=True)
newest_first = prices.sort_values("month", ascending=False).head(4).reset_index(drop=True)
side_by_side = pd.concat([first_rows, newest_first], axis=1)
print(side_by_side)

Elke rij koppelt nu een maand aan de prijs van een andere maand. Het herstel is de tekst omzetten naar een getal, en dan koppelen op de sleutel.

In [ ]:
prices_text["month"] = prices_text["month"].astype(int)
joined = sales.merge(prices_text, on="month", how="left", validate="many_to_one")
print("Rijen na omzetten en koppelen:", len(joined))

## 8. Brede en lange tabellen

Dezelfde getallen kun je op twee manieren ordenen, en de vorm bepaalt welke klussen makkelijk zijn.

- Een **brede** tabel heeft één rij per ding en één kolom per maand of jaar. Een mens leest hem makkelijk, en veel spreadsheets en downloads komen zo binnen.
- Een **lange** tabel heeft één rij per ding per maand of jaar. Daarmee kun je makkelijk koppelen, filteren en rekenen.

De maandelijkse verkoopdoelen van de bakkerij komen breed binnen: één rij per winkel, één kolom per maand.

In [ ]:
targets_wide = pd.DataFrame({"shop": shops["shop"]})
for m in months:
    targets_wide[str(m)] = rng.choice([850, 900, 950], size=len(shops))   # kolomnamen zijn tekst, zoals in een gedownload bestand
targets_wide

`melt` maakt van de brede tabel een lange. De kolomnamen worden waarden in een nieuwe kolom. Kolomnamen zijn altijd tekst, dus de nieuwe maandkolom is ook tekst, en die moet een getal worden voordat je hem aan `sales` kunt koppelen.

In [ ]:
targets = targets_wide.melt(id_vars="shop", var_name="month", value_name="target")
print(targets.dtypes)
targets["month"] = targets["month"].astype(int)
print()
print("Rijen in de lange tabel:", len(targets))
targets.head()

De lange tabel heeft dezelfde granulariteit als `sales`: één rij per winkel per maand. De sleutel bestaat uit twee kolommen, `shop` en `month`, en een koppeling op allebei schrijf je als `on=["shop", "month"]`.

## 9. Jouw beurt: haalde elke winkel zijn doel?

Koppel `targets` aan `sales` op zowel `shop` als `month`. Maak daarna een Series met de naam `hits`, met één waarde per winkel: het aantal maanden waarin de winkel minstens zijn doel verkocht. Neem een winkel ook op als zijn aantal nul is.

Vraag je AI-assistent om de code. Schrijf op hoeveel rijen de koppeling hoort te geven voordat je de code uitvoert, en controleer het aantal daarna.

In [ ]:
# jouw beurt 1: maak een Series met de naam hits, met de winkel als index en het aantal maanden dat elke winkel zijn doel haalde
hits = None

In [ ]:
# CHECK 1
assert hits is not None, "hits is nog leeg"
expected = {}
for s in shops["shop"]:
    n = 0
    for m in months:
        sold = sales.loc[(sales["shop"] == s) & (sales["month"] == m), "loaves"].iloc[0]
        goal = targets.loc[(targets["shop"] == s) & (targets["month"] == m), "target"].iloc[0]
        n += int(sold >= goal)
    expected[s] = n
assert {k: int(v) for k, v in hits.items()} == expected, "de aantallen kloppen niet"
print("CHECK 1 geslaagd:", expected)

## 10. Jouw beurt: omzet per wijk

Breng nu de week samen. Maak twee dingen:

- `district_revenue`, een Series met de wijk als index, met de omzet van januari tot en met november;
- `unpriced`, het aantal verkooprijen zonder prijs.

Gebruik koppelingen die elke verkooprij houden, zet `validate` in beide koppelingen en druk na elke koppeling het aantal rijen af. Vraag je AI-assistent om hulp, en controleer elke koppeling die hij schrijft aan het aantal rijen.

In [ ]:
# jouw beurt 2: maak district_revenue (een Series met de wijk als index) en unpriced (een geheel getal)
district_revenue = None
unpriced = None

In [ ]:
# CHECK 2
assert district_revenue is not None and unpriced is not None, "district_revenue of unpriced is nog leeg"
price_of = dict(zip(prices["month"], prices["price"]))
by_district = {}
for r in sales.merge(shops, on="shop").itertuples():
    if r.month in price_of:
        by_district[r.district] = by_district.get(r.district, 0) + r.loaves * price_of[r.month]
assert set(int(k) for k in district_revenue.index) == set(by_district), "er is één rij per wijk nodig"
assert all(abs(float(district_revenue[k]) - v) < 0.01 for k, v in by_district.items()), "de omzet telt niet op"
assert int(unpriced) == int((~sales["month"].isin(prices["month"])).sum()), "unpriced is niet het aantal rijen zonder prijs"
print("CHECK 2 geslaagd: de omzet telt op tot", f"{float(district_revenue.sum()):,.2f}", "en", int(unpriced), "rijen hebben geen prijs")

## 11. De gewoonte bij elke koppeling

1. Zeg wat één rij betekent in elke tabel, en test of de sleutel uniek is waar dat hoort.
2. Controleer of de sleutels in beide tabellen hetzelfde type en dezelfde schrijfwijze hebben.
3. Kies de soort koppeling op basis van de vraag, en voorspel het aantal rijen.
4. Voer de koppeling uit met `validate`, en met `indicator=True` als sommige rijen misschien geen match hebben.
5. Tel de rijen na de koppeling. Als het aantal afwijkt van je voorspelling, zoek dan uit waarom voordat je iets berekent.